In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats 

In [2]:
# Это мы в Clickhouse ходить будем
import clickhouse_connect
import pandas as pd

# Подключение к ClickHouse
client = clickhouse_connect.get_client(
    host='clickhouse.lab.karpov.courses',
    port=8123,
    username='student',
    password='dpo_python_2020',
    database='simulator_20251120'
)

In [4]:
q = """
select views, count() as users
from (select  
    user_id,
    sum(action = 'view') as views
from simulator_20251120.feed_actions
where toDate(time) between '2025-10-19' and '2025-10-25'
group by user_id
)
group by views
order by views
"""


views_distribution = client.query_df(q)
views_distribution['p'] = views_distribution.users / views_distribution.users.sum()
display(views_distribution)

,views,users,p
0,1,4,0.000095
1,2,1,0.000024
2,3,4,0.000095
3,4,5,0.000119
4,5,18,0.000429
...,...,...,...
296,324,1,0.000024
297,326,1,0.000024
298,355,1,0.000024
299,364,1,0.000024


In [17]:
q = """
select 
   floor(ctr, 2) as ctr, count() as users
from (select toDate(time) as dt, 
    user_id,
    sum(action = 'like')/sum(action = 'view') as ctr
from simulator_20251120.feed_actions
where dt between '2025-10-19' and '2025-10-25'
group by dt, user_id
)
group by ctr
"""


ctr_distribution = client.query_df(q)
ctr_distribution['p'] = ctr_distribution['users']/ctr_distribution.users.sum()
ctr_distribution.sort_values(by = 'ctr', ascending = True)
display(ctr_distribution.sort_values(by = 'ctr', ascending = True))

,ctr,users,p
0,0.00,1443,0.016952
69,0.02,48,0.000564
23,0.03,142,0.001668
32,0.04,312,0.003665
16,0.05,727,0.008541
...,...,...,...
12,0.81,2,0.000023
73,0.83,1,0.000012
26,0.85,3,0.000035
39,0.88,1,0.000012


In [56]:
query = """

select xxHash64(user_id)%2 as bucket, count(user_id) from
(select distinct user_id as user_id
FROM simulator_20251120.feed_actions 
WHERE toDate(time) between '2025-10-19' and '2025-10-25')
group by bucket
"""

df_bucket = client.query_df(query)
display(df_bucket)

,bucket,count(user_id)
0,0,20956
1,1,21041


In [ ]:
# Вариант через цикл


rng = np.random.default_rng()

from tqdm import tqdm

cnt = 0

for i in tqdm(range(20000)):
    group_A_views = rng.choice(views_distribution.views.astype('int64'), size=20000, replace=True, p= views_distribution.p)
    group_B_views = rng.choice(views_distribution.views.astype('int64'), size=20000, replace=True, p= views_distribution.p)
    group_B_views = group_B_views + (rng.binomial(n=1, p=0.5, size=20000) + 1) * rng.binomial(n=1, p=0.9, size=20000) * (group_B_views >= 50)
    group_A_ctr = rng.choice(ctr_distribution.ctr, size=20000, replace=True, p= ctr_distribution.p)
    group_B_ctr= rng.choice(ctr_distribution.ctr, size=20000, replace=True, p= ctr_distribution.p)
    
    clicks_A = rng.binomial(group_A_views, group_A_ctr)
    clicks_B = rng.binomial(group_B_views, group_B_ctr)
    
    if stats.ttest_ind(clicks_A, clicks_B, equal_var=False)[1] <= 0.05:  
        cnt += 1
    
    
print(cnt/20000)
    
    

100%|██████████| 20000/20000 [04:39<00:00, 71.54it/s]

0.25035


In [58]:
# Вариант Яна

group_A_views = rng.choice(views_distribution.views.astype('int64'), size=(20000, 20000), replace=True, p= views_distribution.p)
group_B_views = rng.choice(views_distribution.views.astype('int64'), size=(20000, 20000), replace=True, p= views_distribution.p)
group_B_views = group_B_views + (rng.binomial(n=1, p=0.5, size=(20000, 20000)) + 1) * rng.binomial(n=1, p=0.9, size=(20000, 20000)) * (group_B_views >= 50)
group_A_ctr = rng.choice(ctr_distribution.ctr, size=(20000, 20000), replace=True, p= ctr_distribution.p)
group_B_ctr= rng.choice(ctr_distribution.ctr, size=(20000, 20000), replace=True, p= ctr_distribution.p)

clicks_A = rng.binomial(group_A_views, group_A_ctr)
clicks_B = rng.binomial(group_B_views, group_B_ctr)


def t_test(a, b):
    """
    Считает p-value для t-теста с двусторонней альтернативой 
    :param a: np.array вида (n_experiments, n_users), значения метрик в контрольных группах
    :param b: np.array вида (n_experiments, n_users), значения метрик в тестовых группах
    :return: np.array вида (n_experiments), посчитанные p-value t-теста для всего списка экспериментов
    """
    result = list(map(lambda x: stats.ttest_ind(
        x[0], x[1], equal_var=False).pvalue, zip(a, b)))
    return np.array(result)

print(np.sum(t_test(clicks_A/group_A_views, clicks_B/group_B_views)<= 0.05)/20000)

0.04825
